In [5]:
!pip install sentence-transformers chromadb groq pandas -q
print("Installed library requirements")

Installed library requirements


In [6]:
import pandas as pd #loading and managing college_notes.csv dataset
import chromadb # the vector db library to store document embeddings and perform similarity search
from sentence_transformers import SentenceTransformer #class that loads pre-trained LLM models
from groq import Groq #Groq API client for API calls
import os
from chromadb.utils import embedding_functions


In [7]:
GROQ_API_KEY = "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"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)

print("Initialised GROQ API Key")

Initialised GROQ API Key


In [8]:
df = pd.read_csv("college_notes.csv")
print(df.head())
print()
print(df.columns)

  note_id           subject                     topic  \
0    N001  Data Engineering             ETL Pipelines   
1    N002  Data Engineering             SQL Databases   
2    N003  Data Engineering             Data Cleaning   
3    N004  Data Engineering  APIs and Data Collection   
4    N005  Data Engineering      Big Data and PySpark   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  
3  An API or Application Programming Interface al...  
4  Big Data refers to extremely large datasets th...  

Index(['note_id', 'subject', 'topic', 'content'], dtype='object')


In [9]:
documents = df['content'].to_list()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadata = [
    {'subject': row['subject'], 'topic': row['topic']}
    for row in df.to_dict('records')
]

print(f"Total chunks prepared: {len(documents)}")
print(f"First document ID: {ids[0]}")
print(f"First metadata: {metadata[0]}")
print(f"First document: {documents[0][:100]}...")

Total chunks prepared: 14
First document ID: note_N001
First metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First document: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [10]:
#Loading embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
test_embedding = embedding_model.encode("This is a test sentence.")
print(f"Test embedding shape: {test_embedding.shape}")
print("First 5 values of test embedding:")
print(test_embedding[:5])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Test embedding shape: (384,)
First 5 values of test embedding:
[0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [11]:
#chromaDB
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name='college_notes_rag')
print("chromaDB client created")
print("Collection name: college_notes_rag")
print(f"Documents in collection so far: {collection.count()}")

chromaDB client created
Collection name: college_notes_rag
Documents in collection so far: 0


In [12]:
print("Generating embeddings for all 15 notes...")
embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"\nEmbedding matrix shape: {embeddings.shape}")

embeddings_list = embeddings.tolist()
collection.add(
    documents=documents,
    metadatas=metadata,
    ids=ids,
    embeddings=embeddings_list
)

print("\nDocuments successfully added to chromaDB")
print(f"\nTotal documents in collection: {collection.count()}")


Generating embeddings for all 15 notes...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (14, 384)

Documents successfully added to chromaDB

Total documents in collection: 14


In [13]:
def retrieve_relevant_chunks(question, top_k=3):
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )

  return results

print("Retrival function defined for 3 most relevant notes!")

def build_context_from_results(results):
  context_parts = []

  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    context_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}\n{doc}]"
    context_parts.append(context_text)

    return context_parts
print("Defined build context from results!")
def generate_rag_answer(question, context):
  system_prompt = """Your are a helpful academic assistant for engineering students.

  You will be given context retrieved from a college knowledge base, and a student's question

  RULES:
  1. Answer ONLY using the instruction provided in the context below.
  2. If the answer is not found in the context, say exactly:
      "I don't have enough information in my knowledge base to answer this question."
  3. Do not use your general training knowledge.
  4. Keep answer clear, accurate, and beginner-friendly.
  5. Mention which source the information came from when possible."""
  #USER PROMPT: The context + question formatted
  user_prompt = f"""Context:
  {context}

  Question:
  {question}
  Please answer the question based only on the context provided above."""

  response = groq_client.chat.completions.create(
      model="llama-3.1-8b-instant", #to use same model from training
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      temperature = 0.1,
      #temperature - 0.1 - Very low randomness - we want factual, consistent answers, for RAG, low temp is preffered so the LLM sticks to the context
      max_tokens = 500 #max len of gen responce
  )

  #Extract the text answer from the API resonse object
  answer = response.choices[0].message.content
  #response.choices : A list of responses
  return answer
print("Defined RAG generation function!")

Retrival function defined for 3 most relevant notes!
Defined build context from results!
Defined RAG generation function!


In [14]:
HISTORY = []

In [15]:
student_question = input("Enter your question: ")

# 1. Retrieve relevant chunks
retrieved_results = retrieve_relevant_chunks(student_question, top_k=3)

# 2. Build context from retrieved chunks
context_for_llm = build_context_from_results(results=retrieved_results)

# Join the context parts into a single string for the LLM
formatted_context = "\n\n".join(context_for_llm)

# 3. Generate the answer using the RAG function
final_answer = generate_rag_answer(student_question, formatted_context)

HISTORY.append([student_question, final_answer])

print(f"Question: {student_question}")
print("\n--- Generated Answer ---")
print(final_answer)

Enter your question: is Machine Learning and GenAI the same?
Question: is Machine Learning and GenAI the same?

--- Generated Answer ---
Based on the provided context, it appears that Generative AI (GenAI) and Large Language Models (LLMs) are related concepts, but not exactly the same as Machine Learning.

The context mentions that a Large Language Model is an AI model trained on massive amounts of text data, which is a type of Machine Learning. However, it does not explicitly state that Machine Learning and GenAI are the same.

Therefore, I would say that Machine Learning is a broader field that includes Large Language Models, while GenAI is a specific type of AI model that can perform tasks such as generating human-like text.

Source: [Source 1: Generative AI - Large Language Models]


In [17]:
HISTORY

[['is Machine Learning and GenAI the same?',
  'Based on the provided context, it appears that Generative AI (GenAI) and Large Language Models (LLMs) are related concepts, but not exactly the same as Machine Learning.\n\nThe context mentions that a Large Language Model is an AI model trained on massive amounts of text data, which is a type of Machine Learning. However, it does not explicitly state that Machine Learning and GenAI are the same.\n\nTherefore, I would say that Machine Learning is a broader field that includes Large Language Models, while GenAI is a specific type of AI model that can perform tasks such as generating human-like text.\n\nSource: [Source 1: Generative AI - Large Language Models]']]